# Detective R-GNN — Stage 1: Behavioural Cloning (Warm Start)

Training notebook che imita l'euristica `detective_automated_turn` usando il dataset generato da `colab_warmstart_logger.ipynb`.

**Architettura** (da `DESIGN.md` §6):
- R-GCN dense: 3 layer, hidden 64, residual sul self-loop, ReLU, dropout 0.1
- 4 relazioni: taxi / bus / underground / water
- Policy head edge-conditioned: 1 logit per neighbor legale (no logit su tutti i 199 nodi)
- Value head: mean-pool + max-pool su node embeddings + global features → MLP → scalar

**Loss**: cross-entropy sulla policy (target = `chosen_dest`) + MSE sul value (target = `return_to_go` letto dal parquet, calcolato dal logger con la reward del Design Doc §5).

**Note implementative**:
- Non usiamo PyG: il grafo è fissato a 199 nodi e viene replicato a ogni sample → un'implementazione dense (matmul su matrici di adiacenza precomputate) è ~10× più veloce. PyG rimane utile in Stage 2 (PPO) se si vuole batching dinamico.
- **Nessun leakage**: `mrx_visited` letto dall'NPZ `mrx_visited_history` (solo posizioni rivelate ai turni 3/8/13/18, mai la traiettoria reale di Mr.X).
- `return_to_go` e `all_det_tickets` letti direttamente dal parquet, non ricalcolati.
- `policy_act()` richiede `revealed_positions` esplicito: niente fallback automatico su `game.mrx_moves` (vedi cella inference).

**Pre-requisito dataset**: il parquet `samples.parquet` deve contenere le colonne `all_det_tickets` (lista 15 int) e `return_to_go`, e l'NPZ deve contenere `mrx_visited_history`. Tutti generati dal logger v2 (post-fix reveal turns 3/8/13/18).


## 1. Setup dipendenze + Drive

In [ ]:
!pip install -q networkx numpy pandas pyarrow tqdm

In [ ]:
%%bash
set -e
cd /content
if [ ! -d scotland_yard ]; then
  git clone --depth 1 https://github.com/Jacopo888/scotland_yard.git
fi
ls scotland_yard

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
DRIVE_BASE = '/content/drive/MyDrive/scotland_yard_dataset'
# Auto-discover il run più recente
runs = sorted(glob.glob(os.path.join(DRIVE_BASE, 'run_*')))
assert runs, f'Nessun dataset in {DRIVE_BASE} — runna prima colab_warmstart_logger.ipynb'
DATA_DIR = runs[-1]
print('Using dataset:', DATA_DIR)
for f in os.listdir(DATA_DIR):
    p = os.path.join(DATA_DIR, f)
    print(f'  {f}  {os.path.getsize(p)/(1024*1024):.2f} MB')

CHECKPOINT_DIR = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Imports + repo path

In [ ]:
import os, sys, json, time, math, random, pickle
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

REPO_DIR = '/content/scotland_yard'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

from game import ADJ, BOARD_GRAPH, SHORTEST_PATH_TENSOR, ticket_bitmask

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

torch.manual_seed(0); np.random.seed(0); random.seed(0)

## 3. Static graph: edge_index, dense adjacency per relazione, node static features

Il grafo non cambia mai → precomputiamo le matrici di adiacenza (199×199, row-normalized per relazione) una sola volta. La forward dell'R-GCN diventa una somma di matmul `A_r @ H @ W_r`.

Per-node static features (4 + 1 = 5):
- `degree_taxi/bus/underground/water` (int → float)
- `has_underground` (bool → float)

In [ ]:
N_NODES = 199
RELATIONS = ('taxi', 'bus', 'underground', 'water')
REL2IDX = {r: i for i, r in enumerate(RELATIONS)}
VEHICLES = ('taxi', 'bus', 'underground')   # acquistabili dai detective
VEH2IDX = {v: i for i, v in enumerate(VEHICLES)}


def build_dense_adj():
    """Adiacenza (R, N, N) row-normalized: A_r[v,u] = 1/|N_r(v)| se (v,u) in r."""
    A = np.zeros((len(RELATIONS), N_NODES, N_NODES), dtype=np.float32)
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        for u_str, types in neigh.items():
            u = int(u_str) - 1
            for t in types:
                if t in REL2IDX:
                    A[REL2IDX[t], v, u] = 1.0
    # Row-normalize per relazione
    deg = A.sum(axis=2, keepdims=True)
    deg[deg == 0] = 1.0
    A = A / deg
    return A


def build_node_static_features():
    """(N, 5): degree per relazione + has_underground."""
    deg = np.zeros((N_NODES, len(RELATIONS)), dtype=np.float32)
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        for u_str, types in neigh.items():
            for t in types:
                if t in REL2IDX:
                    deg[v, REL2IDX[t]] += 1.0
    has_und = (deg[:, REL2IDX['underground']] > 0).astype(np.float32)[:, None]
    return np.concatenate([deg, has_und], axis=1)


def build_edge_lookup():
    """Per ogni nodo v: lista di (u, set_of_relation_idx)."""
    lookup = {}
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        out = []
        for u_str, types in neigh.items():
            u = int(u_str) - 1
            rel_idx = sorted({REL2IDX[t] for t in types if t in REL2IDX})
            out.append((u, rel_idx))
        lookup[v] = out
    return lookup


DENSE_ADJ = torch.from_numpy(build_dense_adj()).to(DEVICE)               # (R, N, N)
NODE_STATIC = torch.from_numpy(build_node_static_features()).to(DEVICE)  # (N, 5)
EDGE_LOOKUP = build_edge_lookup()                                        # dict[v_idx] -> [(u_idx, [rel_idx])]

# SHORTEST_PATH_TENSOR ha shape (8, 199, 199): l'asse 0 è il bitmask di tickets.
# Per le features per-nodo (min_dist e dist_to_detective_k) usiamo lo slice 7
# (any-vehicle, idem alla heuristic in _min_detective_distance).
SP_ANY = torch.from_numpy(SHORTEST_PATH_TENSOR[7, :199, :199].astype(np.float32)).to(DEVICE)  # (199, 199)  # SHORTEST_PATH_TENSOR ha shape (8, 200, 200) con riga/col 199 di padding

print('DENSE_ADJ', DENSE_ADJ.shape, 'NODE_STATIC', NODE_STATIC.shape, 'SP_ANY', SP_ANY.shape)
print('Avg deg per relation:', DENSE_ADJ.sum(dim=(1,2)).cpu().numpy() / N_NODES)
# Reveal schedule (allineato a main.py / utility.py: (turn-3) % 5 == 0)
REVEAL_TURNS = (3, 8, 13, 18)

def turns_to_next_reveal(t):
    """Game-turns fino al prossimo reveal di Mr.X. -1 dopo l'ultimo reveal (turno 18)."""
    for r in REVEAL_TURNS:
        if r > t:
            return r - t
    return -1



## 4. Reward function (Design Doc §5) + return-to-go

In [ ]:
GAMMA = 0.95
R_CAPTURE = +10.0
R_TIMEOUT = -10.0
R_STEP = -0.05
R_DIST_COEF = -0.1
MAX_TURN = 22

_SP_ANY_NP = SHORTEST_PATH_TENSOR[7, :199, :199].astype(np.float32)  # trim padding row/col 199


def per_step_reward(mrx_pos_int, detectives_pos_list_int, captured, timeout):
    """Reward computata DOPO il turno (al momento della transizione).
    mrx_pos / detectives_pos sono 1-indexed."""
    if captured:
        terminal = R_CAPTURE
    elif timeout:
        terminal = R_TIMEOUT
    else:
        terminal = 0.0
    mi = mrx_pos_int - 1
    min_d = min(_SP_ANY_NP[p - 1, mi] for p in detectives_pos_list_int)
    return R_STEP + R_DIST_COEF * float(min_d) + terminal


def compute_return_to_go(rewards, gamma=GAMMA):
    R = 0.0
    out = [0.0] * len(rewards)
    for i in reversed(range(len(rewards))):
        R = rewards[i] + gamma * R
        out[i] = R
    return out

## 5. Caricamento dataset + pre-processing per game

In [ ]:
samples_path = os.path.join(DATA_DIR, 'samples.parquet')
games_path = os.path.join(DATA_DIR, 'games.parquet')
tensors_path = os.path.join(DATA_DIR, 'tensors.npz')

df = pd.read_parquet(samples_path)
games_df = pd.read_parquet(games_path)
tens = np.load(tensors_path)

BELIEFS = tens['beliefs']                        # (N_samples, 199) float32
LEGAL_MASKS = tens['legal_masks']                # (N_samples, 199, 3) bool
DET_OH = tens['detectives_one_hot']              # (N_samples, 5, 199) bool
MRX_VISITED_HIST = tens['mrx_visited_history']   # (N_samples, 199) bool — solo posizioni rivelate (turni 3/8/13/18)

print(f'Samples: {len(df)} | Games: {len(games_df)}')
print(f'BELIEFS {BELIEFS.shape} LEGAL_MASKS {LEGAL_MASKS.shape} DET_OH {DET_OH.shape} MRX_VISITED_HIST {MRX_VISITED_HIST.shape}')
print('Cols:', list(df.columns))

assert 'all_det_tickets' in df.columns, 'Dataset deve avere colonna all_det_tickets (rigenera con logger v2)'
assert 'return_to_go' in df.columns, 'Dataset deve avere colonna return_to_go (rigenera con logger v2)'


In [ ]:
# (rimosso) La ricostruzione di all_det_tk_* era necessaria solo per dataset
# pre-patch privi della colonna. Il logger attuale salva 'all_det_tickets'
# come lista flat di 15 int per riga: la usiamo direttamente in cell 16.
print('Skip — all_det_tickets letti dal parquet')


In [ ]:
# (rimosso) Pre-processing per-game.
# - mrx_visited e' gia' in MRX_VISITED_HIST dall'NPZ (solo reveal, no leakage).
# - return_to_go e' gia' calcolato dal logger e salvato nel parquet (consistency).
# Niente da ricalcolare qui.
print('Skip — mrx_visited_history e return_to_go letti direttamente dai file del logger')
print('return_to_go stats:', df['return_to_go'].describe())


## 6. Dataset PyTorch

In [ ]:
# step_idx originale del NPZ (dopo il sort, df.index != step_idx)
df = df.sort_values(['game_id', 'turn', 'detective_id']).reset_index(drop=True)
STEP_IDX = df['step_idx'].to_numpy()

# Pre-cache colonne in numpy per __getitem__ veloce
ARR_CHOSEN_DEST = df['chosen_dest'].to_numpy(dtype=np.int64) - 1     # 0-indexed
ARR_CHOSEN_VEH = df['chosen_vehicle_idx'].to_numpy(dtype=np.int64)
ARR_DET_ID = df['detective_id'].to_numpy(dtype=np.int64)
ARR_TURN = df['turn'].to_numpy(dtype=np.int64)
ARR_MRX_TK_TAXI = df['mrx_tk_taxi'].to_numpy(dtype=np.float32)
ARR_MRX_TK_BUS = df['mrx_tk_bus'].to_numpy(dtype=np.float32)
ARR_MRX_TK_UND = df['mrx_tk_underground'].to_numpy(dtype=np.float32)
ARR_MRX_TK_WAT = df['mrx_tk_water'].to_numpy(dtype=np.float32)
ARR_LAST_TICKET = df['last_mrx_ticket'].to_numpy()
ARR_RTG = df['return_to_go'].to_numpy(dtype=np.float32)  # direttamente dal parquet, no recompute

# all_det_tickets nel parquet e' lista flat di 15 int: (det0_taxi, det0_bus, det0_und, det1_taxi, ...)
ARR_ALL_DET_TK = np.stack([
    np.asarray(x, dtype=np.float32).reshape(5, 3) for x in df['all_det_tickets']
])  # (N, 5, 3)

ARR_DET_POS = np.stack(df['detectives_pos'].tolist()).astype(np.int64) - 1  # (N, 5) 0-indexed
ARR_POS_BEFORE = (df['pos_before'].to_numpy(dtype=np.int64) - 1)            # 0-indexed

# Guard: detectives_pos[detective_id] DEVE coincidere con pos_before per ogni riga.
# Se fallisce, il logger ha salvato detectives_pos post-move (bug noto).
_det_id_arr = df['detective_id'].to_numpy(dtype=np.int64)
_pos_in_det_pos = ARR_DET_POS[np.arange(len(df)), _det_id_arr]
_mismatch = (_pos_in_det_pos != ARR_POS_BEFORE).sum()
assert _mismatch == 0, (
    f'Dataset incoerente: {_mismatch}/{len(df)} righe con detectives_pos[det_id] != pos_before. '
    f'Probabile bug nel logger (detectives_pos salvato POST-move). Rigenerare con logger v3.'
)
print(f'OK: detectives_pos coerente con pos_before su tutte le {len(df)} righe.')

TICKET_VOCAB = {'': 0, 'taxi': 1, 'bus': 2, 'underground': 3, 'water': 4}
ARR_LAST_TICKET_IDX = np.array([TICKET_VOCAB.get(t, 0) for t in ARR_LAST_TICKET], dtype=np.int64)

# Reverse mapping: idx_in_df -> step_idx_in_npz
STEP2NPZ = STEP_IDX.copy()

GLOBAL_FEATURE_DIM = 1 + 1 + 4 + 15 + 5  # turn, turns_to_reveal, mrx_tk(4), all_det_tk(15), last_ticket_one_hot(5)
NODE_DYN_DIM = 1 + 5 + 1 + 1 + 5 + 1   # belief, is_det_k(5), is_ego, min_dist, dist_to_det_k(5), mrx_visited
NODE_FEATURE_DIM = NODE_DYN_DIM + NODE_STATIC.shape[1]  # 14 dyn + 5 static = 19 (matches DESIGN.md §3)
print(f'NODE_FEATURE_DIM={NODE_FEATURE_DIM}  GLOBAL_FEATURE_DIM={GLOBAL_FEATURE_DIM}')


class DetectiveDataset(Dataset):
    def __len__(self):
        return len(df)
    def __getitem__(self, idx):
        npz_i = STEP2NPZ[idx]
        det_id = ARR_DET_ID[idx]
        det_pos = ARR_DET_POS[idx]              # (5,)
        ego_pos = ARR_POS_BEFORE[idx]
        turn = ARR_TURN[idx]

        # ---- Per-node dynamic features ----
        belief = BELIEFS[npz_i]                                              # (199,)
        det_oh = DET_OH[npz_i].astype(np.float32)                            # (5, 199)
        is_det_per_node = det_oh.T                                           # (199, 5)
        is_ego = np.zeros(N_NODES, dtype=np.float32); is_ego[ego_pos] = 1.0
        sp = _SP_ANY_NP
        dist_per_det = np.stack([sp[p] for p in det_pos], axis=1).astype(np.float32)  # (199, 5)
        min_dist = dist_per_det.min(axis=1, keepdims=True)                            # (199, 1)
        # mrx_visited: SOLO posizioni rivelate (turni 3/8/13/18) — niente leakage
        mrx_vis = MRX_VISITED_HIST[npz_i].astype(np.float32)                          # (199,)

        node_dyn = np.concatenate([
            belief[:, None],                # (199, 1)
            is_det_per_node,                # (199, 5)
            is_ego[:, None],                # (199, 1)
            min_dist,                       # (199, 1)
            dist_per_det,                   # (199, 5)
            mrx_vis[:, None],               # (199, 1)
        ], axis=1).astype(np.float32)       # (199, NODE_DYN_DIM)

        legal_mask = LEGAL_MASKS[npz_i]     # (199, 3) bool
        legal_neighbors_mask = legal_mask.any(axis=1)   # (199,) bool

        # ---- Global features ----
        last_ticket_oh = np.zeros(5, dtype=np.float32); last_ticket_oh[ARR_LAST_TICKET_IDX[idx]] = 1.0
        glob = np.concatenate([
            np.array([turn / MAX_TURN, (turns_to_next_reveal(turn) / 5.0)], dtype=np.float32),
            np.array([ARR_MRX_TK_TAXI[idx], ARR_MRX_TK_BUS[idx], ARR_MRX_TK_UND[idx], ARR_MRX_TK_WAT[idx]], dtype=np.float32) / 10.0,
            ARR_ALL_DET_TK[idx].reshape(-1) / 10.0,    # (15,)
            last_ticket_oh,                            # (5,)
        ])

        return {
            'node_dyn': torch.from_numpy(node_dyn),
            'glob': torch.from_numpy(glob),
            'det_id': int(det_id),
            'ego_pos': int(ego_pos),
            'legal_mask': torch.from_numpy(legal_mask),                          # (199, 3)
            'legal_neighbors_mask': torch.from_numpy(legal_neighbors_mask),      # (199,)
            'chosen_dest': int(ARR_CHOSEN_DEST[idx]),                            # 0-indexed
            'rtg': float(ARR_RTG[idx]),
        }


def collate(batch):
    out = {
        'node_dyn': torch.stack([b['node_dyn'] for b in batch]),                # (B, 199, F)
        'glob': torch.stack([b['glob'] for b in batch]),                        # (B, G)
        'det_id': torch.tensor([b['det_id'] for b in batch], dtype=torch.long), # (B,)
        'ego_pos': torch.tensor([b['ego_pos'] for b in batch], dtype=torch.long),
        'legal_mask': torch.stack([b['legal_mask'] for b in batch]),            # (B, 199, 3)
        'legal_neighbors_mask': torch.stack([b['legal_neighbors_mask'] for b in batch]),  # (B, 199)
        'chosen_dest': torch.tensor([b['chosen_dest'] for b in batch], dtype=torch.long),
        'rtg': torch.tensor([b['rtg'] for b in batch], dtype=torch.float),
    }
    return out


ds = DetectiveDataset()
print('Dataset ready, len =', len(ds))


## 7. Modello R-GCN dense + heads

In [ ]:
class DenseRGCNLayer(nn.Module):
    """h' = ReLU(W_0 h + sum_r A_r h W_r) + h   (residual sul self-loop)"""
    def __init__(self, in_dim, out_dim, n_relations, dropout=0.1):
        super().__init__()
        self.W0 = nn.Linear(in_dim, out_dim, bias=True)
        self.Wr = nn.Parameter(torch.empty(n_relations, in_dim, out_dim))
        nn.init.xavier_uniform_(self.Wr)
        self.dropout = nn.Dropout(dropout)
        self.residual = (in_dim == out_dim)
    def forward(self, h, A):
        # h: (B, N, in)  A: (R, N, N)
        self_msg = self.W0(h)                                # (B, N, out)
        # rel_msg per relazione: A_r @ h @ W_r
        # h: (B, N, in) → (B, N, in)  W_r: (in, out)  A_r: (N, N)
        # Useremo einsum per chiarezza
        # AH: (R, B, N, in) = einsum('rmn,bnk->rbmk', A, h)
        AH = torch.einsum('rmn,bnk->rbmk', A, h)             # (R, B, N, in)
        AHW = torch.einsum('rbmk,rkj->rbmj', AH, self.Wr)    # (R, B, N, out)
        rel_msg = AHW.sum(dim=0)                             # (B, N, out)
        out = F.relu(self_msg + rel_msg)
        out = self.dropout(out)
        if self.residual:
            out = out + h
        return out


class DetectiveRGNN(nn.Module):
    def __init__(self, node_dyn_dim, node_static_dim, global_dim,
                 hidden=64, n_layers=3, n_relations=4, det_emb_dim=8,
                 edge_emb_dim=8, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(node_dyn_dim + node_static_dim, hidden)
        self.layers = nn.ModuleList([
            DenseRGCNLayer(hidden, hidden, n_relations, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.det_id_emb = nn.Embedding(5, det_emb_dim)
        self.edge_type_emb = nn.Embedding(n_relations, edge_emb_dim)

        # Policy head: input = [h_v, h_u, edge_emb, global_features+det_emb]
        ph_in = 2*hidden + edge_emb_dim + global_dim + det_emb_dim
        self.policy_mlp = nn.Sequential(
            nn.Linear(ph_in, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

        # Value head: mean+max pool node embeddings + global + det_emb
        vh_in = 2*hidden + global_dim + det_emb_dim
        self.value_mlp = nn.Sequential(
            nn.Linear(vh_in, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def encode(self, node_dyn, node_static_b, A):
        x = torch.cat([node_dyn, node_static_b], dim=-1)   # (B, N, dyn+static)
        h = self.input_proj(x)                              # (B, N, hidden)
        for layer in self.layers:
            h = layer(h, A)                                 # (B, N, hidden)
        return h

    def forward(self, batch, A, node_static):
        B = batch['node_dyn'].shape[0]
        node_dyn = batch['node_dyn']                        # (B, N, dyn_dim)
        glob = batch['glob']                                # (B, global)
        det_id = batch['det_id']                            # (B,)
        ego_pos = batch['ego_pos']                          # (B,)
        legal_neigh = batch['legal_neighbors_mask']         # (B, N) bool

        node_static_b = node_static.unsqueeze(0).expand(B, -1, -1)
        h = self.encode(node_dyn, node_static_b, A)         # (B, N, H)
        det_e = self.det_id_emb(det_id)                     # (B, det_emb_dim)
        glob_full = torch.cat([glob, det_e], dim=-1)        # (B, glob+det_emb)

        # ---- Value head ----
        h_mean = h.mean(dim=1)
        h_max = h.max(dim=1).values
        v_in = torch.cat([h_mean, h_max, glob_full], dim=-1)
        value = self.value_mlp(v_in).squeeze(-1)            # (B,)

        # ---- Policy head: edge-conditioned ----
        # Per ogni sample b: h_v = h[b, ego_pos[b]], poi per ogni neighbor legale u
        # logit_b(u) = MLP([h_v, h_u, edge_emb_b(v→u), glob_full_b])
        # Iteriamo per sample (B di solito 64-256 → loop OK e clearer).
        all_logits = []
        all_chosen_pos = []     # indice nel candidate set del chosen_dest per ogni sample
        chosen_dest = batch['chosen_dest']
        for b in range(B):
            v = int(ego_pos[b].item())
            mask = legal_neigh[b].cpu().numpy() if legal_neigh.is_cuda else legal_neigh[b].numpy()
            # Candidati = (u, edge_type_set) tali che mask[u] == True
            cand = [(u, rels) for (u, rels) in EDGE_LOOKUP[v]
                    if u < N_NODES and mask[u]]
            if not cand:
                # Detective bloccato: nessuna mossa legale (non dovrebbe succedere
                # nel dataset, ma per robustezza emettiamo logit dummy)
                all_logits.append(torch.zeros(1, device=h.device))
                all_chosen_pos.append(0)
                continue
            us = torch.tensor([u for (u, _) in cand], device=h.device, dtype=torch.long)
            # edge embedding: somma sui tipi presenti
            edge_e = torch.stack([self.edge_type_emb(
                torch.tensor(rels, device=h.device, dtype=torch.long)).sum(dim=0)
                for (_, rels) in cand], dim=0)             # (|cand|, edge_emb)
            h_v = h[b, v].unsqueeze(0).expand(len(cand), -1)
            h_u = h[b, us]                                  # (|cand|, H)
            g_b = glob_full[b].unsqueeze(0).expand(len(cand), -1)
            feat = torch.cat([h_v, h_u, edge_e, g_b], dim=-1)
            logits = self.policy_mlp(feat).squeeze(-1)      # (|cand|,)
            all_logits.append(logits)
            # Trova posizione del chosen_dest nei candidati
            try:
                pos = [u for (u, _) in cand].index(int(chosen_dest[b].item()))
            except ValueError:
                pos = 0   # non dovrebbe succedere (verificato in sanity check)
            all_chosen_pos.append(pos)

        # Loss policy: CE su ogni sample (variabile lunghezza)
        nll = h.new_zeros(B)
        for b in range(B):
            logp = F.log_softmax(all_logits[b], dim=-1)
            nll[b] = -logp[all_chosen_pos[b]]
        return value, nll, all_logits, all_chosen_pos


model = DetectiveRGNN(
    node_dyn_dim=NODE_DYN_DIM,
    node_static_dim=NODE_STATIC.shape[1],
    global_dim=GLOBAL_FEATURE_DIM,
    hidden=64, n_layers=3, n_relations=len(RELATIONS),
).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params/1e3:.1f}k params')

## 8. Smoke test: forward + backward su 1 batch

In [ ]:
smoke_loader = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate)
batch = next(iter(smoke_loader))
for k, v in batch.items():
    if torch.is_tensor(v):
        batch[k] = v.to(DEVICE)
value, nll, _, _ = model(batch, DENSE_ADJ, NODE_STATIC)
loss = nll.mean() + 0.05 * F.mse_loss(value, batch['rtg'])  # peso simbolico; il loop reale normalizza rtg
loss.backward()
print(f'OK — value={value[:3].detach().cpu().tolist()} nll={nll[:3].detach().cpu().tolist()} loss={loss.item():.3f}')
model.zero_grad()

## 9. Train/val split per game_id (no leakage)

In [ ]:
all_games = games_df['game_id'].unique().tolist()
rng = np.random.default_rng(0)
rng.shuffle(all_games)
n_val = max(1, int(0.15 * len(all_games)))
val_games = set(all_games[:n_val])
train_games = set(all_games[n_val:])
df_game_id = df['game_id'].to_numpy()
train_idx = np.where(np.isin(df_game_id, list(train_games)))[0]
val_idx = np.where(np.isin(df_game_id, list(val_games)))[0]
print(f'Train: {len(train_idx)} samples ({len(train_games)} games)  |  Val: {len(val_idx)} ({len(val_games)})')

train_loader = DataLoader(Subset(ds, train_idx), batch_size=128, shuffle=True,
                          collate_fn=collate, num_workers=0, drop_last=True)
val_loader = DataLoader(Subset(ds, val_idx), batch_size=256, shuffle=False,
                        collate_fn=collate, num_workers=0)

## 10. Training loop

In [ ]:
EPOCHS = 30
VALUE_WEIGHT = 0.05   # ridotto da 0.5: il value head overfitta dopo epoch 2 ed e' meno importante per Stage 1 BC
LR = 1e-3
WD = 1e-5
EARLY_STOP_PATIENCE = 6  # epoche senza miglioramento di val_nll prima di fermarsi (None per disabilitare)

model = DetectiveRGNN(
    node_dyn_dim=NODE_DYN_DIM,
    node_static_dim=NODE_STATIC.shape[1],
    global_dim=GLOBAL_FEATURE_DIM,
    hidden=64, n_layers=3, n_relations=len(RELATIONS), dropout=0.1,
).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

# Normalizzazione return_to_go sui SOLI dati di training (no leakage dalla val split).
# Aiuta sia la stabilita' della loss sia la calibrazione iniziale dei pesi del value head.
RTG_MEAN = float(ARR_RTG[train_idx].mean())
RTG_STD = float(ARR_RTG[train_idx].std() + 1e-8)
print(f'return_to_go normalization (train split): mean={RTG_MEAN:.3f}  std={RTG_STD:.3f}')
RTG_MEAN_T = torch.tensor(RTG_MEAN, device=DEVICE)
RTG_STD_T = torch.tensor(RTG_STD, device=DEVICE)

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
history = []
best_val_nll = float('inf')
best_path = os.path.join(CHECKPOINT_DIR, f'rgnn_bc_best_{RUN_TAG}.pt')
epochs_since_improve = 0


def run_epoch(loader, train=True):
    model.train(train)
    losses, nlls, mses, accs = [], [], [], []
    pbar = tqdm(loader, desc='train' if train else 'val ', leave=False)
    for batch in pbar:
        for k, v in batch.items():
            if torch.is_tensor(v): batch[k] = v.to(DEVICE, non_blocking=True)
        if train: opt.zero_grad()
        with torch.set_grad_enabled(train):
            value, nll, all_logits, all_chosen_pos = model(batch, DENSE_ADJ, NODE_STATIC)
            loss_pi = nll.mean()
            # Normalizza rtg prima della MSE: la scala del target diventa O(1) e impedisce
            # al value head di dominare il gradiente quando rtg ha range ~[-11, +10].
            rtg_norm = (batch['rtg'] - RTG_MEAN_T) / RTG_STD_T
            loss_v = F.mse_loss(value, rtg_norm)
            loss = loss_pi + VALUE_WEIGHT * loss_v
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        # accuracy top-1 sulla policy
        with torch.no_grad():
            correct = sum(int(torch.argmax(lg).item() == p)
                          for lg, p in zip(all_logits, all_chosen_pos))
            accs.append(correct / len(all_logits))
        losses.append(loss.item()); nlls.append(loss_pi.item()); mses.append(loss_v.item())
        pbar.set_postfix(loss=f'{loss.item():.3f}', acc=f'{accs[-1]:.2f}')
    return np.mean(losses), np.mean(nlls), np.mean(mses), np.mean(accs)


for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_l, tr_nll, tr_mse, tr_acc = run_epoch(train_loader, train=True)
    va_l, va_nll, va_mse, va_acc = run_epoch(val_loader, train=False)
    dt = time.time() - t0
    history.append(dict(epoch=ep,
                        train_loss=tr_l, train_nll=tr_nll, train_mse=tr_mse, train_acc=tr_acc,
                        val_loss=va_l,   val_nll=va_nll,   val_mse=va_mse,   val_acc=va_acc,
                        sec=dt))
    improved = va_nll < best_val_nll
    marker = '  *' if improved else ''
    print(f'Ep {ep:02d}/{EPOCHS}  '
          f'train: nll={tr_nll:.3f} mse={tr_mse:.3f} acc={tr_acc:.3f}  |  '
          f'val: nll={va_nll:.3f} mse={va_mse:.3f} acc={va_acc:.3f}  '
          f'({dt:.1f}s){marker}')
    # Best checkpoint criterion = val_nll. Il value head puo' overfittare senza
    # penalizzare la selezione del checkpoint (era il problema del run precedente).
    if improved:
        best_val_nll = va_nll
        epochs_since_improve = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': ep,
            'val_nll': float(va_nll),
            'val_acc': float(va_acc),
            'val_mse': float(va_mse),
            'rtg_mean': RTG_MEAN,        # serve per de-normalizzare V(s) a inference
            'rtg_std': RTG_STD,
            'config': dict(node_dyn_dim=NODE_DYN_DIM,
                           node_static_dim=int(NODE_STATIC.shape[1]),
                           global_dim=GLOBAL_FEATURE_DIM,
                           hidden=64, n_layers=3, n_relations=len(RELATIONS),
                           gamma=GAMMA, value_weight=VALUE_WEIGHT,
                           best_criterion='val_nll'),
        }, best_path)
        print(f'  ↳ new best (val_nll={va_nll:.3f}, val_acc={va_acc:.3f}), saved to {best_path}')
    else:
        epochs_since_improve += 1
        if EARLY_STOP_PATIENCE is not None and epochs_since_improve >= EARLY_STOP_PATIENCE:
            print(f'Early stop: {EARLY_STOP_PATIENCE} epoche senza miglioramento di val_nll. '
                  f'Best val_nll={best_val_nll:.3f}.')
            break

hist_df = pd.DataFrame(history)
hist_df.to_csv(os.path.join(CHECKPOINT_DIR, f'rgnn_bc_history_{RUN_TAG}.csv'), index=False)
print(f'Done. Best val_nll: {best_val_nll:.3f} (criterion = val_nll, not val_loss).')


## 11. Plot loss / accuracy

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(hist_df['epoch'], hist_df['train_nll'], label='train')
axes[0].plot(hist_df['epoch'], hist_df['val_nll'], label='val')
axes[0].set_title('Policy NLL'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(hist_df['epoch'], hist_df['train_mse'], label='train')
axes[1].plot(hist_df['epoch'], hist_df['val_mse'], label='val')
axes[1].set_title('Value MSE'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True)
axes[2].plot(hist_df['epoch'], hist_df['train_acc'], label='train')
axes[2].plot(hist_df['epoch'], hist_df['val_acc'], label='val')
axes[2].set_title('Top-1 action accuracy'); axes[2].set_xlabel('epoch'); axes[2].legend(); axes[2].grid(True)
plt.tight_layout(); plt.show()

## 12. (Opzionale) Inference helper — usabile in evaluation / Stage 2 PPO

Funzione che dato uno stato di gioco produce la mossa per il detective `det_id`.

In [ ]:
@torch.no_grad()
def policy_act(model, game, engine, det_id, revealed_positions=(), last_mrx_ticket=''):
    """Restituisce (best_dest_int, vehicle_str) usando la GNN al posto dell'euristica.

    Args:
        revealed_positions: iterable di posizioni Mr.X **rivelate** (1-indexed) finora.
            DEVE contenere SOLO i reveal effettivi (turni 3, 8, 13, 18), MAI la traiettoria
            completa di game.mrx_moves: usarla causerebbe information leakage.
        last_mrx_ticket: ultimo ticket usato da Mr.X (osservabile dai detective).

    Returns:
        (best_dest_str, vehicle_str). (None, None) se il detective non ha mosse legali.
    """
    model.eval()
    mrx_visited_mask = np.zeros(N_NODES, dtype=np.float32)
    for p in revealed_positions:
        mrx_visited_mask[int(p) - 1] = 1.0

    det_pos_arr = np.array([int(p) - 1 for p in game.detectives_pos], dtype=np.int64)
    ego = det_pos_arr[det_id]
    belief = engine.belief_state.astype(np.float32)
    is_det = np.zeros((5, N_NODES), dtype=np.float32)
    for k, p in enumerate(det_pos_arr): is_det[k, p] = 1.0
    is_ego = np.zeros(N_NODES, dtype=np.float32); is_ego[ego] = 1.0
    sp = _SP_ANY_NP
    dpd = np.stack([sp[p] for p in det_pos_arr], axis=1).astype(np.float32)
    md = dpd.min(axis=1, keepdims=True)
    node_dyn = np.concatenate([
        belief[:, None], is_det.T, is_ego[:, None], md, dpd, mrx_visited_mask[:, None]
    ], axis=1).astype(np.float32)

    # Costruisci legal mask al volo
    legal = np.zeros((N_NODES, 3), dtype=bool)
    occupied = set(game.detectives_pos[:det_id] + game.detectives_pos[det_id + 1:])
    tickets = game.detective_tickets[det_id]
    for nb_, types in ADJ[game.detectives_pos[det_id]].items():
        if nb_ in occupied: continue
        for t in types:
            if t in VEH2IDX and tickets.get(t, 0) > 0:
                legal[int(nb_) - 1, VEH2IDX[t]] = True
    legal_neigh = legal.any(axis=1)

    # Guard: detective bloccato (nessuna mossa legale)
    cand = [u for (u, _) in EDGE_LOOKUP[ego] if legal_neigh[u]]
    if not cand:
        return None, None

    # Globals
    last_oh = np.zeros(5, dtype=np.float32); last_oh[TICKET_VOCAB.get(last_mrx_ticket, 0)] = 1.0
    all_dt = np.array([[t['taxi'], t['bus'], t['underground']] for t in game.detective_tickets],
                      dtype=np.float32) / 10.0
    glob = np.concatenate([
        np.array([game.turn / MAX_TURN, (turns_to_next_reveal(game.turn) / 5.0)], dtype=np.float32),
        np.array([game.mrx_tickets[v] for v in ('taxi','bus','underground','water')],
                 dtype=np.float32) / 10.0,
        all_dt.reshape(-1),
        last_oh,
    ])

    batch = collate([{
        'node_dyn': torch.from_numpy(node_dyn),
        'glob': torch.from_numpy(glob),
        'det_id': det_id,
        'ego_pos': int(ego),
        'legal_mask': torch.from_numpy(legal),
        'legal_neighbors_mask': torch.from_numpy(legal_neigh),
        'chosen_dest': 0,
        'rtg': 0.0,
    }])
    for k, v in batch.items():
        if torch.is_tensor(v): batch[k] = v.to(DEVICE)
    _, _, all_logits, _ = model(batch, DENSE_ADJ, NODE_STATIC)
    logits = all_logits[0]
    best_idx = int(torch.argmax(logits).item())
    best_dest = cand[best_idx] + 1   # 1-indexed (string-friendly per il game)
    # vehicle scelto: priority taxi -> bus -> underground (come l'euristica use_ticket)
    chosen_v = next(v for v in ('taxi','bus','underground') if legal[best_dest - 1, VEH2IDX[v]])
    return str(best_dest), chosen_v

print('policy_act() pronto. Esempio (revealed_positions=() = nessun reveal ancora):')
from game import Game
from detective_engine import DetectiveEngine
_g = Game(); _e = DetectiveEngine(_g.detectives_pos)
print(policy_act(model, _g, _e, det_id=0, revealed_positions=()))
